In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

In [20]:
from tensorflow.keras.datasets import mnist
(x_train , y_train), (x_test, y_test) = mnist.load_data()
x_train = torch.from_numpy(np.array(x_train, dtype = 'float32') / 255.0)
x_test  = torch.from_numpy(np.array(x_test, dtype = 'float32') / 255.0)

y_train = torch.from_numpy(y_train).type(torch.LongTensor)
y_test = torch.from_numpy(y_test).type(torch.LongTensor)

In [21]:
batch_size = 16
train_set = TensorDataset(x_train, y_train)
test_set = TensorDataset(x_test, y_test)
train_loader = DataLoader(
    train_set,
    batch_size = batch_size,
    shuffle = True
)
test_loader = DataLoader(
    test_set,
    batch_size = batch_size,
    shuffle = False
)

In [22]:
# Khởi tạo mô hình BiRNN
class BiRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(BiRNNModel, self).__init__()
        
        # Thiết lập số chiều của tầng ẩn
        self.hidden_dim = hidden_dim
        
        # Thiết lập số layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity='relu', bidirectional=True)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
    
    def forward(self, x):
        
        # Khởi tạo hidden state
        h0 = torch.zeros((self.layer_dim * 2, x.size(0), self.hidden_dim)).to('cuda')
        # print(1)
        # One time step
        out, hn = self.rnn(x, h0)
        # print(2)
        # print(out.shape)
        out = self.fc(out[:, -1, :]) 
        # print(3)
        return out

In [23]:
input_dim = 28
hidden_dim = 100
layer_dim = 3
output_dim = 10

model = BiRNNModel(input_dim = input_dim, hidden_dim = hidden_dim, layer_dim = layer_dim, output_dim = output_dim).to('cuda')
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)
criterion = nn.CrossEntropyLoss()


In [24]:
for epoch in range(5):
    model.train()
    
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        images, targets = data
        images = images.to('cuda')
        targets = targets.to('cuda')

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets) 

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        if i % 1000 == 999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

[1,  1000] loss: 1.151
[1,  2000] loss: 1.149
[1,  3000] loss: 1.116
[2,  1000] loss: 0.402
[2,  2000] loss: 0.255
[2,  3000] loss: 0.185
[3,  1000] loss: 0.128
[3,  2000] loss: 0.112
[3,  3000] loss: 0.100
[4,  1000] loss: 0.083
[4,  2000] loss: 0.074
[4,  3000] loss: 0.076
[5,  1000] loss: 0.063
[5,  2000] loss: 0.060
[5,  3000] loss: 0.059


In [30]:
correct = 0
total = 0
with torch.no_grad():
    model.eval()
    for data in test_loader:
        images, labels = data
        images = images.to('cuda')
        labels = labels.to('cuda')

        outputs = model(images)
        predict = torch.argmax(outputs, dim = 1) 
        correct += (predict == labels).sum().item()
        total += labels.size(0)
        accuracy = correct / total * 100
    print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 96.77%
